In [1]:
#Pré-processamento: foco na preservação e integridade estatística

In [2]:
import pandas as pd
import numpy as np
import os

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split



In [3]:
#caminhos
train_path = "../raw_data/UNSW_NB15_training-set.csv"
test_path  = "../raw_data/UNSW_NB15_testing-set.csv"

#garantia de existência
assert os.path.exists(train_path)
assert os.path.exists(test_path)

#leitura dos dados
#usando Pandas para ler aquivos .csv e transformá-los em dataframes
train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

#dimensões
print("Exemplos para treino e característcas:",train_df.shape,"\nExemplos para teste e características:", test_df.shape)



Exemplos para treino e característcas: (175341, 45) 
Exemplos para teste e características: (82332, 45)


In [4]:
#essencial a remoção do ID para o modelo não sofrer Overfitting de Identidade, decorando
#o registro do ataque. Aqui forço o modelo a olhar o comportamento real do ataque.
drop_cols = ["id"]

#remoção do tipo de ataque. O modelo é um Classificador Binário, quero saber se é ou não um ataque
#além disso, há possibilidade de haver Data Leakege ("roubo" de resposta)
if "attack_cat" in train_df.columns:
    drop_cols.append("attack_cat")

#aplicando as mesmas configs nos dois conjuntos
train_df = train_df.drop(columns=drop_cols)
test_df  = test_df.drop(columns=drop_cols)



In [5]:
#quero prever a coluna 'label': 0 -> tráfego normal; 1 -> ataque (independente de qual seja)
target = "label"

#conjunto de características (x): removo a coluna de respostas para o modelo não ter acesso durante o treinamento
#vetor de respostas (y): variáveis recebem apenas 'label'. Uso values para trabalhar com matrizes numéricas à tabelas Pandas.
X_train_df = train_df.drop(columns=[target])
y_train = train_df[target].values

X_test_df = test_df.drop(columns=[target])
y_test = test_df[target].values



In [6]:
#identifiquei manualmente as variáveis categóricas
cat_cols = ["proto", "service", "state"]

#identificação automática das variáveis numéricas
num_cols = [c for c in X_train_df.columns if c not in cat_cols]

print("Numéricas:", len(num_cols))
print("Categóricas:", cat_cols)


Numéricas: 39
Categóricas: ['proto', 'service', 'state']


In [7]:
#uso do One-Hot-Encoding para transformar cada categoria em uma nova coluna de binários
ohe = OneHotEncoder(
    sparse_output=False, # força o resultado a ser uma matriz densa comum. Garanto que os dados estão prontos para virarem Tensores
    handle_unknown="ignore" #modelo ignora informação que ele não reconhece (evita erros)
)

X_train_cat = ohe.fit_transform(X_train_df[cat_cols]) #codificador aprende quais categorias existem e crias as colunas
X_test_cat  = ohe.transform(X_test_df[cat_cols]) #usa apenas as colunas que aprendeu no treino

#aqui o número de colunas tende a aumentar, pois as colunas categóricas têm muitas variações, e cada variação vira uma coluna nova
print("One-hot shape:", X_train_cat.shape)


One-hot shape: (175341, 155)


In [8]:
#OHE: cuida dos textos
#StandardScaler: cuida dos números
#uso o standardscaler para colocar as colunas na mesma "escala", transformando os números para que
#a média seja 0 e o desvio padrão 1
#o modelo não deve achar que um número é mais importante que outro por ser "maior"

scaler = StandardScaler()

X_train_num = scaler.fit_transform(X_train_df[num_cols]) #fit: calcula a médias e o desvio padrão de cada coluna do conjunto de treino;
X_test_num  = scaler.transform(X_test_df[num_cols]) #transform: aplica a fórmula e transforma os dados de treino

#basicamente garanto que o modelo trate todas as informações numéricas com a mesma importância, permitindo aprendizado
#rápido e estável


In [9]:
#fusão final dos dados com empilhamento horizontal
#foi necessário fazer isso uma vez que, uma MLP precisa de um único tensor (vetor de entrada)

X_train_final = np.hstack([X_train_num, X_train_cat])
X_test_final  = np.hstack([X_test_num, X_test_cat])

print("Número de conexões de rede (linhas) e soma das colunas originais com as novas (train shape):", X_train_final.shape)
print("Número de conexões de rede (linhas) e soma das colunas originais com as novas (test shape):", X_test_final.shape)

#esta abordagem é essencial para converter os dados em tensores

Número de conexões de rede (linhas) e soma das colunas originais com as novas (train shape): (175341, 194)
Número de conexões de rede (linhas) e soma das colunas originais com as novas (test shape): (82332, 194)


In [10]:
#crio um diretório

os.makedirs("../artifacts/data", exist_ok=True)


In [11]:
#salvo os dados corretamente

np.save("../artifacts/data/X_train.npy", X_train_final)
np.save("../artifacts/data/X_test.npy", X_test_final)

np.save("../artifacts/data/y_train.npy", y_train)
np.save("../artifacts/data/y_test.npy", y_test)

print("Arquivos salvos com sucesso.")


Arquivos salvos com sucesso.
